In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================

# 1. Install Unsloth (lets Unsloth manage compatible versions)
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. RAG / Vector Database
!pip install -U chromadb sentence-transformers

# 3. Data Processing
!pip install -U datasets langchain-text-splitters

# 4. Restart runtime after installation


In [ ]:
import unsloth

from unsloth import FastLanguageModel
from transformers import TrainingArguments
from peft import LoraConfig
import os
import csv
import re
import math
import logging
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import torch
import numpy as np
import chromadb
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import logging as hf_logging
from transformers import AutoTokenizer


# ==========================================
# PERFORMANCE SETTINGS
# ==========================================
torch.backends.cuda.matmul.allow_tf32 = True
hf_logging.set_verbosity_error()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ==========================================
# CONFIGURATION
# ==========================================
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_CONTEXT_TOKENS = 2048
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16

RERANKER_MODEL = "ncbi/MedCPT-Cross-Encoder"
INITIAL_DENSE_TOP_K = 10
INITIAL_BM25_TOP_K  = 10
FINAL_TOP_K = 5
ALPHA_DENSE = 0.6
ALPHA_BM25  = 0.4

# ==========================================
# SYSTEM PROMPTS
# ==========================================
SYS_PROMPT_MEDQA_NO_RAG = (
    "You are an expert medical AI assistant. Your task is to answer the user's question. "
    "Formulate a detailed explanation, and conclude with a final decision of A, B, C, or D."
)

SYS_PROMPT_MEDQA_RAG = (
    "You are an expert medical AI assistant. You will be provided with medical abstracts as context. "
    "Your task is to carefully read the context and use it to answer the user's question. "
    "Formulate a detailed explanation based on the context, and conclude with a final decision of A, B, C, or D."
)

SYS_PROMPT_PUBMED_NO_RAG = (
    "You are a helpful and expert medical assistant. Identify the correct response employing a logical and systematic strategy. "
    "Evaluate the medical premise and conclude your reasoning with a final classification of 'yes', 'no', or 'maybe'."
)

SYS_PROMPT_PUBMED_RAG = (
    "You are a helpful and expert medical assistant. Identify the correct response employing a logical and systematic strategy. "
    "Carefully use the provided research context to evaluate the question. Conclude your reasoning with a final classification of 'yes', 'no', or 'maybe'."
)

In [ ]:
class VectorDB:
    def __init__(self, db_path: str = "./chroma_db"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer("NeuML/pubmedbert-base-embeddings", device=self.device)
        self.client = chromadb.PersistentClient(path=db_path)

        self.collection = self.client.get_or_create_collection(
            name="MedRAG_Hybrid",
            metadata={
                "description": "Medical textbooks and PubMed embeddings",
                "hnsw:space": "cosine"
            }
        )

        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )

    def process_dataset(self, repo_id: str, text_col: str = "contents", title_col: str = "title", max_samples: int = None):
        print(f"-> Đang tải {repo_id}...")
        dataset = load_dataset(repo_id, split="train", streaming=True)

        if max_samples:
            dataset = dataset.take(max_samples)

        chunks_data = []
        for idx, row in enumerate(tqdm(dataset, desc=f"Cắt chunk {repo_id}", total=max_samples)):
            text = row.get(text_col, "")
            title = row.get(title_col, "Unknown Title")
            doc_id = str(row.get("id", idx))

            if not text:
                continue

            chunks = self.text_splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                chunks_data.append({
                    "id": f"{repo_id}_{doc_id}_chunk_{i}",
                    "text": chunk,
                    "metadata": {
                        "title": title,
                        "source": repo_id,
                        "parent_id": doc_id,
                        "chunk_index": i
                    }
                })
        return chunks_data

    def build_db(self, batch_size: int = 32):
        all_chunks = []
        
        # Original textbooks
        all_chunks.extend(self.process_dataset("MedRAG/textbooks"))
        
        # New Medical Q&A Dataset (16k)
        all_chunks.extend(self.process_dataset(
            "keivalya/MedQuad-MedicalQnADataset", 
            text_col="Answer", 
            title_col="Question"
        ))
        
        # New Medical WikiDoc Dataset (10k)
        all_chunks.extend(self.process_dataset(
            "medalpaca/medical_meadow_wikidoc", 
            text_col="output", 
            title_col="input"
        ))

        total_chunks = len(all_chunks)
        print(f"\n=> Tổng số chunks cần embedding: {total_chunks}")

        for i in tqdm(range(0, total_chunks, batch_size), desc="Đang Embedding & Upsert"):
            batch = all_chunks[i : i + batch_size]

            texts = [item["text"] for item in batch]
            ids = [item["id"] for item in batch]
            metadatas = [item["metadata"] for item in batch]

            embeddings = self.model.encode(
                texts,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

            self.collection.upsert(
                documents=texts,
                embeddings=embeddings.tolist(),
                ids=ids,
                metadatas=metadatas
            )

    def search(self, query, top_k=3):
        query_embeddings = self.model.encode(
            query,
            normalize_embeddings=True,
        ).tolist()

        results = self.collection.query(
            query_embeddings=[query_embeddings],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
        return results


In [ ]:
class BM25Index:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.corpus: List[str] = []
        self.tokenized: List[List[str]] = []
        self.doc_freqs: List[Counter] = []
        self.idf: Dict[str, float] = {}
        self.avgdl: float = 0.0
        self.N: int = 0

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        text = text.lower()
        text = re.sub(r"[^\w\s]", " ", text)
        return [t for t in text.split() if len(t) > 1]

    def fit(self, corpus: List[str]) -> None:
        self.corpus = corpus
        self.N = len(corpus)
        self.tokenized = [self._tokenize(doc) for doc in corpus]
        self.doc_freqs = [Counter(tokens) for tokens in self.tokenized]

        lengths = [len(t) for t in self.tokenized]
        self.avgdl = sum(lengths) / self.N if self.N else 1.0

        df: Counter = Counter()
        for tf in self.doc_freqs:
            for term in tf:
                df[term] += 1

        self.idf = {}
        for term, freq in df.items():
            self.idf[term] = math.log((self.N - freq + 0.5) / (freq + 0.5) + 1)

    def get_scores(self, query: str) -> np.ndarray:
        query_terms = self._tokenize(query)
        scores = np.zeros(self.N)

        for term in query_terms:
            if term not in self.idf:
                continue
            idf = self.idf[term]
            for i, tf in enumerate(self.doc_freqs):
                freq = tf.get(term, 0)
                dl = len(self.tokenized[i])
                numerator   = freq * (self.k1 + 1)
                denominator = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[i]  += idf * numerator / denominator

        return scores

    def search(self, query: str, top_k: int = 20) -> List[Tuple[int, float]]:
        scores = self.get_scores(query)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

class HybridRetriever:
    def __init__(self, vector_db: VectorDB, reranker_model: str = RERANKER_MODEL):
        self.vector_db = vector_db
        self._bm25: Optional[BM25Index] = None
        self._bm25_corpus: List[Dict[str, Any]] = []   
        self._reranker: Optional[CrossEncoder] = None
        self._reranker_model = reranker_model

        self.RRF_THRESHOLD = 0.001
        self.CE_THRESHOLD = 0

    def build_bm25_index(self) -> None:
        logger.info("🔨 Đang build BM25 index từ ChromaDB ...")
        coll = self.vector_db.collection 
        total = coll.count()
        if total == 0:
            logger.warning("⚠️  ChromaDB rỗng — BM25 index không có dữ liệu.")
            return

        FETCH_BATCH = 5000
        all_docs: List[Dict[str, Any]] = []
        offset = 0

        while offset < total:
            result = coll.get(limit=FETCH_BATCH, offset=offset, include=["documents", "metadatas"])
            for doc_id, text, meta in zip(result["ids"], result["documents"], result["metadatas"]):
                all_docs.append({"id": doc_id, "text": text, "metadata": meta})
            offset += FETCH_BATCH

        self._bm25_corpus = all_docs
        corpus_texts = [d["text"] for d in all_docs]

        self._bm25 = BM25Index()
        self._bm25.fit(corpus_texts)
        logger.info(f"✅ BM25 index sẵn sàng: {len(all_docs)} documents.")

    def _get_reranker(self) -> CrossEncoder:
        if self._reranker is None:
            logger.info(f"🔄 Đang tải Cross-Encoder: {self._reranker_model} ...")
            self._reranker = CrossEncoder(self._reranker_model, max_length=512)
            logger.info("✅ Cross-Encoder sẵn sàng.")
        return self._reranker

    def _dense_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        raw_results = self.vector_db.search(query, top_k=top_k)
        results = []
        if not raw_results["documents"] or not raw_results["documents"][0]:
            return results
            
        for i in range(len(raw_results["documents"][0])):
            distance = raw_results["distances"][0][i]
            # print("===== DENSE SEARCH INFO =====")
            similarity_score = 1.0 - distance 
            # print(distance)
            # print(similarity_score)
            # print("="*10)
            
            results.append({
                "id": raw_results["ids"][0][i],
                "text": raw_results["documents"][0][i],
                "metadata": raw_results["metadatas"][0][i],
                "score": similarity_score
            })
        return results

    def _bm25_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if self._bm25 is None or not self._bm25_corpus:
            logger.warning("⚠️  BM25 chưa được build. Gọi build_bm25_index() trước.")
            return []

        ranked = self._bm25.search(query, top_k=top_k)
        results = []
        for idx, score in ranked:
            # print("===== BM25 SEARCH INFO =====")
            # print("Score: {}".format(score))
            # print(self._bm25_corpus[idx])
            # print("="*10)
            if score <= 0:
                continue
            doc = self._bm25_corpus[idx]
            results.append({"id": doc["id"], "text": doc["text"], "metadata": doc["metadata"], "score": score})
        return results

    def _hybrid_merge(self, dense_hits: List[Dict[str, Any]], bm25_hits:  List[Dict[str, Any]], rrf_k: int = 60) -> List[Dict[str, Any]]:
        merged: Dict[str, Dict[str, Any]] = {}

        for rank, hit in enumerate(dense_hits, start=1):
            doc_id = hit["id"]
            rrf_score = 1.0 / (rrf_k + rank)
            weighted_score = ALPHA_DENSE * rrf_score
            merged[doc_id] = {**hit, "hybrid_score": weighted_score}

        for rank, hit in enumerate(bm25_hits, start=1):
            doc_id = hit["id"]
            rrf_score = 1.0 / (rrf_k + rank)
            weighted_score = ALPHA_BM25 * rrf_score
            if doc_id in merged:
                merged[doc_id]["hybrid_score"] += weighted_score
            else:
                merged[doc_id] = {**hit, "hybrid_score": weighted_score}

        sorted_hits = sorted(merged.values(), key=lambda x: x["hybrid_score"], reverse=True)
        
        # print("==== HYBRID INFO ====")
        # print([x["hybrid_score"] for x in sorted_hits])
        # print("="*10)

        filtered_hits = [hit for hit in sorted_hits if hit["hybrid_score"] >= self.RRF_THRESHOLD]

        if filtered_hits:
            logger.info(
                f"Top hybrid score: {filtered_hits[0]['hybrid_score']:.5f}"
            )
        else:
            logger.info(
                f"No candidates passed RRF threshold ({self.RRF_THRESHOLD})"
            )
    
        return filtered_hits        

    def _rerank(
        self,
        query: str,
        candidates: List[Dict[str, Any]],
        top_k: int = FINAL_TOP_K,
        ) -> List[Dict[str, Any]]:
        
        if not candidates:
            return []
        
        reranker = self._get_reranker()
        # print("==== RERANK CONFIG ====")
        # print(reranker.model.config)

        # tokenizer = AutoTokenizer.from_pretrained(
        #     "ncbi/MedCPT-Cross-Encoder"
        # )
        
        # for i, doc in enumerate(candidates[:5]):
        #     n_tokens = len(tokenizer.tokenize(doc["text"]))
        #     print(f"Doc {i}: {n_tokens} tokens")
        # print("="*50)
        
        pairs = [(query, c["text"]) for c in candidates]
        
        ce_scores = reranker.predict(
            pairs,
            show_progress_bar=False
        )
        
        for candidate, ce_score in zip(candidates, ce_scores):
            candidate["rerank_score"] = float(ce_score)
        
        reranked = sorted(
            candidates,
            key=lambda x: x["rerank_score"],
            reverse=True
        )
        # print("==== RERANK INFO ====")
        # print([doc["rerank_score"] for doc in reranked])
        # print("="*10)
        # Filter by CE threshold
        filtered = [
            doc
            for doc in reranked
            if doc["rerank_score"] >= self.CE_THRESHOLD
        ]
        
        logger.info(
            f"Rerank kept {len(filtered)}/{len(reranked)} docs "
            f"(threshold={self.CE_THRESHOLD})"
        )
        
        return filtered[:top_k]
        
    def retrieve(self, query: str, final_top_k: int = FINAL_TOP_K, use_rerank: bool = True) -> List[Dict[str, Any]]:
        logger.info(f"🔍 Hybrid Retrieval cho query: '{query[:80]}...'")
        # print("ENTER RETRIEVE")
        dense_hits = self._dense_search(query, top_k=INITIAL_DENSE_TOP_K)
        bm25_hits = self._bm25_search(query, top_k=INITIAL_BM25_TOP_K)
        merged = self._hybrid_merge(dense_hits, bm25_hits)

        if use_rerank and merged:
            final = self._rerank(query, merged, top_k=final_top_k)
        else:
            final = merged[:final_top_k]
        return final

def build_retriever() -> HybridRetriever:
    db = VectorDB()
    existing_count = db.collection.count()
    if existing_count == 0:
        logger.info("-> Database Chroma rỗng! Đang tiến hành tạo dữ liệu VectorDB (Chỉ chạy 1 lần)...")
        db.build_db(batch_size=32)
    else:
        logger.info(f"-> Dữ liệu VectorDB đã tồn tại ({existing_count} chunks). Bỏ qua tạo mới.")

    retriever = HybridRetriever(db)
    retriever.build_bm25_index()
    # print("Created retriever")
    return retriever

In [ ]:
def extract_mcq_answer(llm_output):
    match_mcq = re.search(r"(?:Answer:|answer is|correct option is|choice is|Conclusion:)\s*([A-D])", llm_output, re.IGNORECASE)
    if match_mcq:
        return match_mcq.group(1).upper()
    match_mcq_end = re.search(r"\b([A-D])\b[\.\s]*(?:<\|im_end\|>)?$", llm_output, re.IGNORECASE)
    if match_mcq_end:
        return match_mcq_end.group(1).upper()
    return None

def extract_ynm_answer(llm_output):
    match_ynm = re.search(r"(?:Answer:|answer is|correct option is|choice is)\s*(yes|no|maybe)", llm_output, re.IGNORECASE)
    if match_ynm:
        return match_ynm.group(1).lower()
    match_ynm = re.search(r"\b(yes|no|maybe)\b[\.\s]*(?:<\|im_end\|>)?$", llm_output, re.IGNORECASE)
    if match_ynm:
        return match_ynm.group(1).lower()
    return None

def format_context(hits):
    if not hits:
        return "No reference context found."
    docs = []
    for i, hit in enumerate(hits):
        title = hit.get("metadata", {}).get("title", f"Document {i+1}")
        text = hit.get("text", "")
        docs.append(f"[Document {i+1} - {title}]\n{text}")
    return "\n\n".join(docs)

def build_messages(user_prompt_text, sys_prompt_no_rag, sys_prompt_rag, tokenizer, use_rag=False, raw_context=""):
    sys_prompt = sys_prompt_rag if use_rag else sys_prompt_no_rag
    if not use_rag or not raw_context:
        return [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt_text}
        ]

    sys_tokens = len(tokenizer.encode(sys_prompt))
    q_tokens = len(tokenizer.encode(user_prompt_text))
    available_context_tokens = MAX_CONTEXT_TOKENS - sys_tokens - q_tokens - 50

    if available_context_tokens <= 0:
        user_content = f"Context:\nNone\n\n{user_prompt_text}"
    else:
        context_tokens = tokenizer.encode(raw_context)
        if len(context_tokens) > available_context_tokens:
            truncated_context = tokenizer.decode(context_tokens[:available_context_tokens])
        else:
            truncated_context = raw_context
        user_content = f"Context:\n{truncated_context}\n\n{user_prompt_text}"

    # print(user_content)

    return [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_content}
    ]

def generate_batch(model, tokenizer, prompts):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_CONTEXT_TOKENS,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1] 

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.05,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[:, input_length:] 
    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return [text.strip() for text in decoded]

In [ ]:
def run_medqa_eval(dataset, model, tokenizer, retriever_engine, use_rag, results_list):
    correct = 0
    total = len(dataset)
    print(f"\n[*] Evaluating MedQA ({total} questions)...")
    progressbar = tqdm(total=total, desc="MedQA", colour="cyan")

    for start_idx in range(0, total, BATCH_SIZE):
        batch = dataset[start_idx:start_idx + BATCH_SIZE]
        prompts, actual_answers, questions, contexts = [], [], [], []

        for question, options, actual in zip(batch["question"], batch["options"], batch["answer_idx"]):
            raw_context = ""
            options_text = "\n".join([f"{k}) {v}" for k, v in options.items()])

            if use_rag and retriever_engine is not None:
                retrieval_query = (
                    f"Question: {question}\n"
                    f"Options:\n{options_text}"
                )
                
                hits = retriever_engine.retrieve(retrieval_query, final_top_k=3, use_rerank=True)
                raw_context = format_context(hits)

            user_prompt_text = f"Question:\n{question}\n\nOptions:\n{options_text}"

            messages = build_messages(user_prompt_text, SYS_PROMPT_MEDQA_NO_RAG, SYS_PROMPT_MEDQA_RAG, tokenizer, use_rag, raw_context)
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            prompts.append(prompt)
            actual_answers.append(actual)
            questions.append(question)
            contexts.append(raw_context)

        generated_outputs = generate_batch(model, tokenizer, prompts)

        for i, generated_text in enumerate(generated_outputs):
            pred = extract_mcq_answer(generated_text)
            is_correct = (str(pred).upper() == str(actual_answers[i]).upper())
            
            if is_correct: correct += 1

            results_list.append({
                "Dataset": "MedQA",
                "Index": start_idx + i + 1,
                "Question": questions[i],
                "Context": contexts[i] if use_rag else "N/A",
                "Expected": actual_answers[i],
                "Predicted": pred if pred else "[Invalid]",
                "IsCorrect": is_correct
            })

        progressbar.update(len(batch["question"]))
    progressbar.close()
    return correct, total


def run_pubmedqa_eval(dataset, model, tokenizer, retriever_engine, use_rag, results_list):
    correct = 0
    total = len(dataset)
    print(f"\n[*] Evaluating PubMedQA ({total} questions)...")
    progressbar = tqdm(total=total, desc="PubMedQA", colour="green")

    for start_idx in range(0, total, BATCH_SIZE):
        batch = dataset[start_idx:start_idx + BATCH_SIZE]
        prompts, actual_answers, questions, contexts = [], [], [], []

        for question, actual in zip(batch["question"], batch["final_decision"]):
            raw_context = ""
            if use_rag and retriever_engine is not None:
                hits = retriever_engine.retrieve(question, final_top_k=3, use_rerank=True)
                raw_context = format_context(hits)

            user_prompt_text = f"Question: {question}\nAnswer:"
            messages = build_messages(user_prompt_text, SYS_PROMPT_PUBMED_NO_RAG, SYS_PROMPT_PUBMED_RAG, tokenizer, use_rag, raw_context)
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            prompts.append(prompt)
            actual_answers.append(actual)
            questions.append(question)
            contexts.append(raw_context)

        generated_outputs = generate_batch(model, tokenizer, prompts)

        for i, generated_text in enumerate(generated_outputs):
            pred = extract_ynm_answer(generated_text)
            is_correct = (pred == actual_answers[i])
            
            if is_correct: correct += 1

            results_list.append({
                "Dataset": "PubMedQA",
                "Index": start_idx + i + 1,
                "Question": questions[i],
                "Context": contexts[i] if use_rag else "N/A",
                "Expected": actual_answers[i],
                "Predicted": pred if pred else "[Invalid]",
                "IsCorrect": is_correct
            })

        progressbar.update(len(batch["question"]))
    progressbar.close()
    return correct, total

In [ ]:
# ==========================================
# KAGGLE CONFIGURATION VARIABLES
# (Replaces argparse for notebook usage)
# ==========================================
# Set LORA_PATH = '' to run the pure base model (no fine-tuning)
# Set LORA_PATH to your CoT-200 adapter path for the fine-tuned model
LORA_PATH = ''  # e.g. '/kaggle/input/my-cot200-lora' or leave empty for base model
USE_RAG = False  # Non-RAG phase: model answers from its own knowledge only
DATASET_LIMIT = 0  # Set to e.g., 50 for a quick test run
OUTPUT_FILE = '/kaggle/working/benchmark_results_no_rag.csv'
# ==========================================

print('=' * 60)
print('MEDICAL BENCHMARK  (Non-RAG Phase)')
print('=' * 60)

# No retriever needed for non-RAG run
retriever_engine = None

print('\n[+] Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_CONTEXT_TOKENS,
    dtype=torch.float16,
    load_in_4bit=True,
)
tokenizer.padding_side = 'left'

if LORA_PATH and os.path.exists(LORA_PATH):
    print(f'[+] Loading LoRA: {LORA_PATH}')
    model.load_adapter(LORA_PATH)
elif LORA_PATH:
    print(f'[-] LoRA path not found: {LORA_PATH}. Running Base Model.')
else:
    print('[+] No LoRA path set — running pure Base Model.')

FastLanguageModel.for_inference(model)
model.eval()
print('[+] Model ready.')

print('\n[+] Loading datasets...')
medqa_ds = load_dataset('GBaker/MedQA-USMLE-4-options', split='test')
pubmedqa_ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')

if DATASET_LIMIT > 0:
    medqa_ds = medqa_ds.select(range(min(DATASET_LIMIT, len(medqa_ds))))
    pubmedqa_ds = pubmedqa_ds.select(range(min(DATASET_LIMIT, len(pubmedqa_ds))))

print(f'  MedQA    : {len(medqa_ds)} questions')
print(f'  PubMedQA : {len(pubmedqa_ds)} questions')


In [ ]:
# ==========================================
# TEST FIRST 5 SAMPLES BEFORE FULL EVALUATION
# ==========================================
test_limit = min(5, len(medqa_ds))
print(f"Testing the first {test_limit} samples to view generation format...")

test_batch = medqa_ds.select(range(test_limit))
test_prompts = []

for question, options in zip(test_batch["question"], test_batch["options"]):
    options_text = "\n".join([f"{k}) {v}" for k, v in options.items()])
    raw_context = ""
    if USE_RAG and retriever_engine is not None:
        retrieval_query = f"Question: {question}\nOptions:\n{options_text}"
        hits = retriever_engine.retrieve(retrieval_query, final_top_k=3, use_rerank=True)
        raw_context = format_context(hits)

    user_prompt_text = f"Question:\n{question}\n\nOptions:\n{options_text}"
    messages = build_messages(user_prompt_text, SYS_PROMPT_MEDQA_NO_RAG, SYS_PROMPT_MEDQA_RAG, tokenizer, USE_RAG, raw_context)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    test_prompts.append(prompt)

test_outputs = generate_batch(model, tokenizer, test_prompts)

for i, out in enumerate(test_outputs):
    print(f"========== SAMPLE {i+1} ==========")
    print("GENERATED ANSWER:")
    print(out)
    print("-" * 40)
    print("EXTRACTED (REGEX):", extract_mcq_answer(out))
    print("==================================\n")


In [ ]:
results_data = []

med_correct, med_total = run_medqa_eval(medqa_ds, model, tokenizer, retriever_engine, USE_RAG, results_data)
med_acc = (med_correct / med_total * 100) if med_total > 0 else 0

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"MedQA Accuracy: {med_acc:.2f}%")
print("=" * 60)

# pub_correct, pub_total = run_pubmedqa_eval(pubmedqa_ds, model, tokenizer, retriever_engine, USE_RAG, results_data)
# pub_acc = (pub_correct / pub_total * 100) if pub_total > 0 else 0

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
# print(f"PubMedQA Accuracy: {pub_acc:.2f}%")
print("=" * 60)

print(f"\n[+] Writing results to {OUTPUT_FILE}")
with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["Dataset", "Correct", "Total", "Accuracy"])
    writer.writerow(["MedQA", med_correct, med_total, f"{med_acc:.2f}%"])
    # writer.writerow(["PubMedQA", pub_correct, pub_total, f"{pub_acc:.2f}%"])

print("[+] Done.")